<a href="https://colab.research.google.com/github/elettra-ven/DIA-Ventura-SolAq/blob/main/SolAq_DIA_Ventura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predizione della Solubilita Acquosa di Molecole Organiche

**Programmazione di Applicazioni Data Intensive**

Ventura Elettra - 0001117272

a.a. 2025/2026


##Obiettivo
Il progetto ha come obiettivo la **predizione della solubilità acquosa* (LogS)* di molecole organiche, a partire dalle loro proprieta fisicochimiche strutturali, utilizzando tecniche di machine learning supervisato.

##0. Import librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

##1. Contesto e Dataset
### 1.1 Introduzione al problema

La **solubilita acquosa** è una proprieta fondamentale per lo studio bioinformatico delle molecole.

La solubilita e espressa come **LogS** (log10 della solubilita in mol/L):
- LogS >= 0      -> altamente solubile
- 0 > LogS >= -2 -> solubile
- -2 > LogS >= -4 -> poco solubile
- LogS < -4      -> insolubile

### 1.2 Dataset e fonti

**AqSolDB** (Kaggle:https://www.kaggle.com/datasets/sorkun/aqsoldb-a-curated-aqueous-solubility-dataset) è stato pubblicato da Sorkun et al. su *Scientific Data* nel 2019.

È costruito fondendo **9 dataset sperimentali** con un algoritmo di curation che seleziona il valore piu affidabile.
- **Dimensione:** 9.982 molecole uniche
- **Target:** LogS - valore sperimentale di solubilita acquosa

In [ ]:
!pip install kaggle
!kaggle datasets download -d sorkun/aqsoldb-a-curated-aqueous-solubility-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/sorkun/aqsoldb-a-curated-aqueous-solubility-dataset
License(s): CC0-1.0
100% 1.39M/1.39M [00:00<00:00, 156MB/s]



In [ ]:
df = pd.read_csv("curated-solubility-dataset.csv")
print("Righe: ",df.shape[0])
print("Colonne: ",df.shape[1])
df.head()

Righe:  9982
Colonne:  26


,ID,Name,InChI,InChIKey,SMILES,Solubility,SD,Ocurrences,Group,MolWt,...,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT
0,A-3,"N,N,N-trimethyloctadecan-1-aminium bromide",InChI=1S/C21H46N.BrH/c1-5-6-7-8-9-10-11-12-13-...,SZEMGTQCPRNXEG-UHFFFAOYSA-M,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127,0.0,1,G1,392.510,...,17.0,142.0,0.0,0.0,0.0,0.0,0.00,158.520601,0.000000e+00,210.377334
1,A-4,Benzo[cd]indol-2(1H)-one,InChI=1S/C11H7NO/c13-11-8-5-1-3-7-4-2-6-9(12-1...,GPYLCFQEKPUWLD-UHFFFAOYSA-N,O=C1Nc2cccc3cccc1c23,-3.254767,0.0,1,G1,169.183,...,0.0,62.0,2.0,0.0,1.0,3.0,29.10,75.183563,2.582996e+00,511.229248
2,A-5,4-chlorobenzaldehyde,InChI=1S/C7H5ClO/c8-7-3-1-6(5-9)2-4-7/h1-5H,AVPYQKSLYISFPO-UHFFFAOYSA-N,Clc1ccc(C=O)cc1,-2.177078,0.0,1,G1,140.569,...,1.0,46.0,1.0,0.0,0.0,1.0,17.07,58.261134,3.009782e+00,202.661065
3,A-8,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",InChI=1S/2C23H22O3.Zn/c2*1-15(17-9-5-3-6-10-17...,XTUPUYCJWKHGSW-UHFFFAOYSA-L,[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409,0.0,1,G1,756.226,...,10.0,264.0,6.0,0.0,0.0,6.0,120.72,323.755434,2.322963e-07,1964.648666
4,A-9,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,InChI=1S/C25H30N2O4/c1-5-20(26(10-22-14-28-22)...,FAUAZXVRLVIARB-UHFFFAOYSA-N,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065,0.0,1,G1,422.525,...,12.0,164.0,2.0,4.0,4.0,6.0,56.60,183.183268,1.084427e+00,769.899934


###1.3 Descrizione delle variabili
TO-DO

##1.4 Prima scrematura dei dati
Rimuoviamo le colonne che non contengono informazioni predittive ("meta-dati" e identificatori):
- `ID`, `Name`, `InChI`, `InChIKey`, `SMILES`: sono tutti **identificatori** e non portano particolari informazioni sulla struttura
- `SD`: **deviazione standard** delle misurazioni effettuate da diverse fonti
- `Occurrences`: numero di occorrenze di un composto
- `Group`: etichetta meta-dato per l'affidabilità

In [ ]:
cols_to_drop = ["ID", "Name", "InChI", "InChIKey", "SMILES",
                "SD", "Ocurrences", "Group"]

df_clean = df.drop(columns=cols_to_drop)

print("Righe dopo scrematura: ",df_clean.shape[0])
print("Colonne dopo scrematura: ",df_clean.shape[1])

Righe dopo scrematura:  9982
Colonne dopo scrematura:  18


##2. Analisi Esplorativa del Dataset (EDA)
###2.1 Statistiche descrittive
###2.2 Verifica valori nulli
###2.3 Distribuzione del target
###2.4 Distribuzione delle feature
####2.4.1 Feature discrete
####2.4.2 Feature continue
###2.5